# Step 7: Concept map

Which measurements exist at all three sites, in which tables, for how many stays, and in what range. `concepts.py` holds the map (`CONCEPTS`: for each concept, its source tables and columns at each site) and measures every source in the rebuilt assembled tables.

**What this notebook does**
1. Builds the concept detail (one row per concept, site and source: stays, rows, % numeric, 5th, median and 95th percentile) and the concept summary (one row per concept: stays and % of each site's stays), using only the stays in the pooled static table.
2. Selects the concepts present at all three sites with at least 50% of each site's stays, which is the common feature set.
3. Builds the stay-by-concept availability table (does each stay have at least one value of each concept), kept in `derived/` because it has one row per stay.
4. Writes a counts-only report to `reports/`.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.home() / "Documents/DS217/users/jormond/open_picu/code"))
import inventory as I, diagnostics as D, concepts as C, anonymise as AN

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")  # keep whatever the assemble notebook uses
derived = open_picu_dir / "derived"
out_dir = derived / "assembled"
static_dir = derived / "static"
concepts_dir = derived / "concepts"
reports_dir = open_picu_dir / "reports"
pooled = pd.read_parquet(static_dir / "pooled_static.parquet")
pooled.groupby("site").size()

## 1. Measure every source
`C.build` reads each source table in the rebuilt assembled folder and, for each concept and site, counts the stays with a value, the rows, the share of values that are numeric, and the 5th, median and 95th percentile. Percentiles make unit differences between sites visible. Only the stays in the pooled static table are counted, and the percentages use that site's number of stays as the denominator. The two tables are saved to `derived/concepts/`: `concept_detail.csv` and `concept_summary.csv`. They hold counts and ranges only. This is the slow cell, because it reads most of the assembled tables.

In [ ]:
detail, summary = C.build(out_dir, static=pooled, save_to=concepts_dir / "concept_detail.csv")
summary

## 2. Concepts common to all three sites
The concepts present at all three sites with at least 50% of each site's stays. Drugs are excluded because their coverage is the share of stays that received the drug, not how well it is recorded. The second table also drops the concepts whose GOSH source ends at the Epic switch in April 2019, which is the set that can be used for the whole period.

In [ ]:
common = C.survivors(summary, min_pct=50)
common_all_years = C.survivors(summary, min_pct=50, exclude_era=True)
len(common), len(common_all_years)

## 3. Which stays have which concepts
`C.availability` gives one row per stay and one column per concept: True if the stay has at least one value, False if the site records the concept but the stay has none, and missing if the site has no source for the concept at all. It is row-level, so it stays in `derived/concepts/`, and it lets users choose a feature set and the stays that have it. The drugs are left out of this table.

In [ ]:
avail = C.availability(out_dir, pooled, save_to=concepts_dir / "stay_availability.parquet")
avail.shape

## 4. Save the report
A counts-only summary goes to `reports/`: how many concepts are mapped, the common sets, and the summary table with stay counts under 10 shown as `<10`.

In [ ]:
shown = summary.drop(columns=["note"]).copy()
for c in [c for c in shown.columns if c.endswith("_stays")]:
    shown[c] = shown[c].map(lambda v: v if pd.isna(v) or v >= 10 else "<10")
D.write_report({
    "Concepts mapped": summary.groupby("group").size().to_frame("concepts"),
    "Common to all three sites (at least 50% of stays)": common[["concept", "group", "RBH_pct", "SMH_pct", "GOSH_pct"]],
    "Common, GOSH source continues after the Epic switch": common_all_years[["concept", "group"]],
    "Summary": shown,
}, reports_dir / f"Concepts_{pd.Timestamp.today().date()}.md", title="Concept map")

## 5. Concept presence by site
One row per concept: 1 if the site has a mapped source for it, 0 if not. This is the mapping, not a count of stays. The workbook goes to `reports/`. The release columns (In release, Released as, Why not in release) are added after the anonymisation step, when the level is known.

In [ ]:
presence = C.write_presence_workbook(reports_dir / "concept_presence_by_site.xlsx")
presence["Sites with the concept"].value_counts().sort_index()

In [ ]:
# should eventually move anonimyze.ipynb
import anonymise as AN

out_root = derived / "anonymised"
cfg = AN.config_from_level(out_root, "tier2_merge_drop3_pre_epic")
h = "gave the site away (" + ", ".join(AN.SITE_PATTERN_DROP) + " dropped by hand)"
presence = C.write_presence_workbook(
    reports_dir / "concept_presence_by_site.xlsx",
    release=dict(drop=cfg.drop_concepts, rename=cfg.rename_concepts, by_hand={c: h for c in AN.SITE_PATTERN_DROP}),
    level="tier2_merge_drop3_pre_epic")
presence[["Concept", "In release (1/0)"]].groupby("In release (1/0)").size()

# Concept map: coverage, feature availability and re-identification risk of availability patterns

Step 7 of the pipeline. Takes the assembled tables and the pooled static table, and measures for every concept (feature) in `concepts.CONCEPTS` how many stays at each site have a value. Then:

1. **Coverage** per concept, site and source (`build`), saved with a dated copy.
2. **Feature sets**: which concepts survive a coverage threshold (`survivors`).
3. **Availability table**: one row per stay, one column per concept (row-level, so kept in `derived/` only).
4. **Pattern uniqueness**: how identifying the availability pattern is (`pattern_uniqueness`, `rare_share`).
5. **Range review**: p5 / median / p95 side by side across sites to spot unit differences (`compare_ranges`).

Drugs are mapped as presence only and are excluded from the survival rule and the availability patterns (they describe exposure, not recording quality). The investigations behind the issues in `DATA_ISSUES.md` are in `concepts_checks.ipynb`.

Only counts and ranges are written to `reports/`. Nothing here leaves the SDE without output checking.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")

assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
derived_dir = open_picu_dir / "derived"
static_dir = open_picu_dir / "derived" / "static"
static = pd.read_parquet(static_dir / "pooled_static.parquet")
print(static.shape, static["site"].value_counts().to_dict())
today = date.today().isoformat()
print(C.__file__, len(C.CONCEPTS), "concepts")

## 2. Feature sets by coverage threshold
A concept survives if at least `min_pct` % of stays have it at **every** site. "Without GOSH Epic-limited" drops the 19 concepts that end at the CareVue to Epic switch (about 2019-04-19) and so are missing in the Epic years (about 42% of GOSH stays). Drug concepts are excluded.

In [ ]:
rows = []
for t in (90, 70, 50):
    rows.append({"min_pct": t,
                 "features": len(C.survivors(summary, min_pct=t)),
                 "without_GOSH_epic_limited": len(C.survivors(summary, min_pct=t, exclude_era=True))})
pd.DataFrame(rows)

## 3. Availability table (row-level: `derived/` only)
One row per stay, one column per non-drug concept. `True` = the stay has a value, `False` = collected at the site but absent for this stay, `<NA>` = the site does not collect it. Users choose their feature set and then the stays that have it, so we keep every stay and publish availability instead of dropping stays.

In [ ]:
avail = C.availability(assembled_dir, static)
avail.to_parquet(Path(derived_dir) / "availability.parquet")
avail.shape

## 4. How identifying is the availability pattern?
A stay's pattern of available concepts is itself a fingerprint. `pattern_uniqueness` counts distinct patterns and the stays whose pattern is shared by fewer than 10 stays. The second table collapses concepts into five groups (vitals, vent, blood gas, labs, fluids), which is the coarser version we could publish. `rare_share` breaks the rare-pattern share down by site and by site and year, since rare patterns concentrate in recent years.

In [ ]:
print(C.pattern_uniqueness(avail))
print(C.pattern_uniqueness(avail, by_group=True))
C.rare_share(avail, static, by=("site",))

In [ ]:
C.rare_share(avail, static, by=("site", "year"))   # use your static table's year column name

### Rare patterns among the surviving features
The figures above use every non-drug concept, including sparse ones (CVP, glucose, INR). Users will pick a feature set, so the reference figure is the share of stays in rare patterns when only the features that survive the 50% threshold are used. This is the number carried in `ASSEMBLY.md` section 5 item 8. The second version also drops the 19 GOSH Epic-limited features, so the Epic-era stays are not penalised for a gap in recording.

In [ ]:
keep = C.survivors(summary, min_pct=50)["concept"].tolist()
keep_era = C.survivors(summary, min_pct=50, exclude_era=True)["concept"].tolist()
for name, cols in (("50% threshold", keep), ("50% threshold, without GOSH Epic-limited", keep_era)):
    print(f"{name}: {len(cols)} features")
    print(C.pattern_uniqueness(avail[cols]))
    print(C.rare_share(avail[cols], static, by=("site",)), "\n")

## 5. Range review across sites
Side-by-side p5 / median / p95 for each numeric concept and site (source with most stays). `unit_flag` is set when the largest median is more than 3 times the smallest, or the sign differs. Flags are reviewed by hand: most are case-mix or real practice differences, and the real unit problems are listed in `DATA_ISSUES.md` and corrected at harmonisation.

In [ ]:
ranges = C.compare_ranges(detail)
ranges[ranges["unit_flag"]]

## 6. Reference numbers
Compare with `ASSEMBLY.md` section 5 item 8. If these change, update that table and the working position.

In [ ]:
print(pd.DataFrame(rows).to_string(index=False))
print("rare share by site:\n", C.rare_share(avail, static, by=("site",)))

## 7. Save the results for review
Writes the reference numbers, coverage summary, pattern uniqueness, rare-pattern shares and range flags to one markdown file in `reports/`. Counts and ranges only; no row-level data.

In [ ]:
def _txt(x):
    return x.to_string() if hasattr(x, "to_string") else str(x)

ref = pd.DataFrame([{"min_pct": t,
                     "features": len(C.survivors(summary, min_pct=t)),
                     "without_GOSH_epic_limited": len(C.survivors(summary, min_pct=t, exclude_era=True))}
                    for t in (90, 70, 50)])

sections = {
    "Reference numbers (features surviving each threshold)": ref,
    "Pattern uniqueness (all concepts)": C.pattern_uniqueness(avail),
    "Pattern uniqueness (by group)": C.pattern_uniqueness(avail, by_group=True),
    "Rare-pattern share by site": C.rare_share(avail, static, by=("site",)),
    "Rare-pattern share by site and year": C.rare_share(avail, static, by=("site", "year")),
    "Coverage summary (stays and % per site)": summary,
    "Range flags (unit_flag)": C.compare_ranges(detail).query("unit_flag"),
}
out = Path(reports_dir) / f"concepts_review_{today}.md"
out.write_text("\n\n".join(f"## {k}\n\n```\n{_txt(v)}\n```" for k, v in sections.items()))
print("written:", out, f"({out.stat().st_size // 1024} KB)")

## E. Save the results for review
Runs the two outstanding checks (GOSH `Total_fluids` and `Net_Bal` by year) and writes the output to one markdown file in `reports/`. Add further calls to `calls` to capture more.

In [ ]:
import io, contextlib
from datetime import date
today = date.today().isoformat()

bal = C._BAL
calls = {
    "GOSH Total_fluids by year": lambda: C.names_by_year(assembled_dir, "GOSH", bal + "Total_fluids", "shortLabel", "start_datetime"),
    "GOSH Net_Bal by year":      lambda: C.names_by_year(assembled_dir, "GOSH", bal + "Net_Bal", "shortLabel", "start_datetime"),
}
parts = []
for name, f in calls.items():
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            r = f()
        text = buf.getvalue() + (r.to_string() if hasattr(r, "to_string") else ("" if r is None else str(r)))
    except Exception as e:
        text = f"ERROR: {type(e).__name__}: {e}"
    parts.append(f"## {name}\n\n```\n{text}\n```")

out = Path(reports_dir) / f"checks_review_{today}.md"
out.write_text("\n\n".join(parts))
print("written:", out)